In [0]:
# Questo notebook prepara la struttura base del progetto Meteo ed Energia.
# Crea il catalogo principale, gli schemi Medallion e le tabelle Delta usate dalla pipeline.
#
# Il progetto usa questa organizzazione:
# - metadati: informazioni statiche, come zone ENTSO-E e città monitorate
# - bronze: dati grezzi o quasi grezzi letti dalle API
# - silver: dati consolidati o aggregati per l'uso operativo
# - gold: dati finali usati per analisi, studio e visualizzazioni
#
# Nota:
# la tabella metadati.citta_monitorate viene popolata nel notebook Setup_Città.
# Qui viene creata la struttura principale delle tabelle operative già usate dal progetto.


# Imposto i nomi principali del progetto.
catalogo = "progetto_meteo"
schema_metadati = "metadati"
schema_bronze = "bronze"
schema_silver = "silver"
schema_gold = "gold"



spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo}")
spark.sql(f"USE CATALOG {catalogo}")


# Creo gli schemi principali del progetto.
# Ogni schema rappresenta un livello logico dell'architettura.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_metadati}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_bronze}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_silver}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_gold}")


# Creo la tabella metadati per le zone ENTSO-E.
# Questa tabella collega ogni zona energetica italiana alla macroarea finale del progetto.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.{schema_metadati}.zone_entsoe (
    Zona STRING,
    Area STRING,
    Codice_Entsoe STRING
)
USING DELTA
""")


# Creo la tabella Bronze per lo storico Open-Meteo.
# Qui finiscono i dati meteorologici storici scaricati dal notebook Bootstrapper.
# La temperatura storica viene salvata come Temp_Oraria.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.{schema_bronze}.meteo_storico (
    Citta STRING,
    Regione STRING,
    Area STRING,
    Data DATE,
    Ora BIGINT,
    Temp_Max DOUBLE,
    Temp_Min DOUBLE,
    Temp_Oraria DOUBLE,
    Umidita DOUBLE,
    Velocita_Vento DOUBLE,
    Precipitazioni DOUBLE,
    Timestamp TIMESTAMP
)
USING DELTA
""")


# Creo la tabella Bronze per i dati live Open-Meteo.
# Qui finiscono le acquisizioni periodiche del notebook LiveForecast.
# La temperatura live viene salvata come Temp_Istantanea perché non è ancora mediata su base oraria.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.{schema_bronze}.meteo_streaming (
    Citta STRING,
    Regione STRING,
    Area STRING,
    Data DATE,
    Ora BIGINT,
    Temp_Max DOUBLE,
    Temp_Min DOUBLE,
    Temp_Istantanea DOUBLE,
    Umidita DOUBLE,
    Velocita_Vento DOUBLE,
    Precipitazioni DOUBLE,
    Timestamp TIMESTAMP
)
USING DELTA
""")


# Creo la tabella Bronze per i dati ENTSO-E.
# Qui vengono salvati i dati energetici scaricati via API prima dell'aggregazione.
# Gli orari UTC vengono mantenuti insieme agli orari convertiti in Europe/Rome.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.{schema_bronze}.dati_entsoe (
    Zona STRING,
    Area STRING,
    Codice_Entsoe STRING,
    Tipo_Produzione STRING,
    Start_UTC TIMESTAMP,
    Fine_UTC TIMESTAMP,
    Start TIMESTAMP,
    Fine TIMESTAMP,
    Data DATE,
    Ora BIGINT,
    Produzione DOUBLE,
    Timestamp_Caricamento TIMESTAMP
)
USING DELTA
""")


# Creo la tabella Silver meteo.
# Questa è la tabella meteo consolidata a livello città, data e ora.
# Viene inizializzata dal Clonatore e poi aggiornata ogni giorno dal Patcher.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.{schema_silver}.dati_aggiornati (
    Citta STRING,
    Regione STRING,
    Area STRING,
    Data DATE,
    Ora BIGINT,
    Temp_Max DOUBLE,
    Temp_Min DOUBLE,
    Temp_Oraria DOUBLE,
    Umidita DOUBLE,
    Velocita_Vento DOUBLE,
    Precipitazioni DOUBLE,
    Timestamp TIMESTAMP
)
USING DELTA
""")


# Creo la tabella Silver energia.
# Qui i dati ENTSO-E vengono aggregati per Area, Data e Ora.
# Le fonti energetiche finali sono Solare, Idrico ed Eolico.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.{schema_silver}.energy_block (
    Area STRING,
    Data STRING,
    Ora BIGINT,
    Solare DOUBLE,
    Idrico DOUBLE,
    Eolico DOUBLE
)
USING DELTA
""")


# Creo la tabella Gold meteo aggregata.
# Qui i dati meteo passano dal livello città al livello macroarea.
# Questa tabella prepara la parte meteo per il join finale con l'energia.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.{schema_gold}.dati_aggregati (
    Area STRING,
    Data STRING,
    Ora BIGINT,
    Temperatura DOUBLE,
    Umidita DOUBLE,
    Velocita_Vento DOUBLE,
    Precipitazioni DOUBLE
)
USING DELTA
""")


# Creo la tabella Gold finale del progetto.
# Questa tabella unisce meteo ed energia per Area, Data e Ora.
# È la base usata per analisi, casi studio e visualizzazioni.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.{schema_gold}.dati_studio (
    Area STRING,
    Data STRING,
    Ora BIGINT,
    Temperatura DOUBLE,
    Umidita DOUBLE,
    Velocita_Vento DOUBLE,
    Precipitazioni DOUBLE,
    Solare DOUBLE,
    Idrico DOUBLE,
    Eolico DOUBLE
)
USING DELTA
""")


# Controllo finale della struttura creata.
# Mostro gli schemi e le tabelle disponibili nei vari layer.
display(spark.sql(f"SHOW SCHEMAS IN {catalogo}"))
display(spark.sql(f"SHOW TABLES IN {catalogo}.{schema_metadati}"))
display(spark.sql(f"SHOW TABLES IN {catalogo}.{schema_bronze}"))
display(spark.sql(f"SHOW TABLES IN {catalogo}.{schema_silver}"))
display(spark.sql(f"SHOW TABLES IN {catalogo}.{schema_gold}"))